# LACT_sim ROOT event through pylast

This notebook checks one LACT_sim ROOT event with the normal pylast flow. It reads the ROOT file once with `LactEventSource`, keeps the same in-memory `event`, then plots array/core, SDP planes, raw camera images, cleaned DL1 images with Hillas ellipses, and prints DL2 reconstruction values.

The plotting calls below pass `event` plus an `EventVisualizer`; they do not re-open the ROOT file, so `ImageProcessor(event)` and `ShowerProcessor(event)` changes are visible in later cells.


In [ ]:
from pathlib import Path
import os
import time

# Keep matplotlib/cache writes away from AFS/home quota on the server.
os.environ.setdefault("MPLCONFIGDIR", "/home/lhaaso/huangyiyun/tmp/matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

# Change this if your run_corsika_trace output went somewhere else.
ROOT_FILE = Path("/home/lhaaso/huangyiyun/LACT/Sim_program/LACT_sim/run_logs/lact_root_only_full_response/lact_events.root")
if not ROOT_FILE.exists():
    ROOT_FILE = Path("/home/lhaaso/huangyiyun/LACT/Sim_program/LACT_sim/run_logs/lact_root_full_response/lact_events.root")

EVENT_INDEX = 0
MAX_EVENTS = 10
IMAGE_LEVEL = "dl0"

print("ROOT_FILE:", ROOT_FILE)
print("exists:", ROOT_FILE.exists())


In [ ]:
import numpy as np

from pylast.io import LactEventSource
from pylast.image import ImageProcessor
from pylast.reco import ShowerProcessor
from pylast.visualize import (
    EventVisualizer,
    hillas_parameter_rows,
    plot_event_cameras,
    plot_event_cores,
    plot_event_sdp_planes,
    reconstruction_summary,
)


def step(name, fn):
    print(f"BEGIN {name}", flush=True)
    t0 = time.perf_counter()
    out = fn()
    print(f"END {name}: {time.perf_counter() - t0:.3f}s", flush=True)
    return out


## 1. Read One Event

This is the only ROOT read in the notebook. Later cells use the same `event` object.


In [ ]:
source = step("LactEventSource", lambda: LactEventSource(str(ROOT_FILE), max_events=MAX_EVENTS))
event = step(f"read event {EVENT_INDEX}", lambda: source[EVENT_INDEX])
visualizer = EventVisualizer(source)

triggered_tels = list(getattr(event.simulation, "triggered_tels", []))
print("event_id:", event.event_id)
print("run_id:", event.run_id)
print("subarray ntel:", len(source.subarray.tels))
print("triggered telescope ids:", triggered_tels)
print("DL0 telescope ids:", sorted(event.dl0.tels.keys()))
print("R1 telescope ids:", sorted(event.r1.tels.keys()))


In [ ]:
tel_id = triggered_tels[0] if triggered_tels else sorted(event.dl0.tels.keys())[0]
dl0_camera = event.dl0.tels[tel_id]
r1_camera = event.r1.tels[tel_id]

print("tel_id:", tel_id)
print("DL0 image shape:", dl0_camera.image.shape)
print("DL0 peak_time shape:", dl0_camera.peak_time.shape)
print("R1 waveform shape:", r1_camera.waveform.shape)
print("R1 gain_selection shape:", r1_camera.gain_selection.shape)
print("DL0 total p.e.:", float(np.sum(dl0_camera.image)))
print("R1 waveform total p.e.:", float(np.sum(r1_camera.waveform)))


## 2. Array/Core Plot

Same role as the old `plot_event_core` / `visualize_telpos` style plot: LACT telescope layout, LHAASO ED/MD background, true shower core, arrival direction, telescope pointing direction, and triggered telescope outlines.


In [ ]:
core_result = plot_event_cores(
    event,
    visualizer=visualizer,
    image_level=IMAGE_LEVEL,
    include_non_triggered=False,
)
core_result["figure"]


## 3. SDP Planes

This plot uses the same event object and overlays the triggered telescope SDP plane ground projections on the core plot. If later we add a 3D SDP view, it can use the same `event` and `visualizer` pattern.


In [ ]:
sdp_result = plot_event_sdp_planes(
    event,
    visualizer=visualizer,
    image_level=IMAGE_LEVEL,
    include_non_triggered=False,
)
sdp_result["figure"]


## 4. Triggered DL0 Camera Images

This is the integrated p.e. image from LACT_sim before pylast image cleaning.


In [ ]:
dl0_result = plot_event_cameras(
    event,
    visualizer=visualizer,
    image_level="dl0",
    include_non_triggered=False,
    show_hillas=False,
)
dl0_result["figure"]


## 5. Image Cleaning And Hillas

This runs pylast `ImageProcessor` on the same event object. The next plot reads `event.dl1` directly and overlays the Hillas ellipses already computed by pylast.


In [ ]:
image_processor = step("ImageProcessor init", lambda: ImageProcessor(source.subarray))
step("image_processor(event)", lambda: image_processor(event))

print("DL1 telescope ids:", sorted(event.dl1.tels.keys()))
rows = hillas_parameter_rows(event)
print("Hillas telescope ids:", [row["tel_id"] for row in rows])
for row in rows:
    print(
        f"tel {row['tel_id']:2d}: intensity={row['intensity']:.2f}, "
        f"length={row['length_rad']:.5g} rad, width={row['width_rad']:.5g} rad, "
        f"psi={np.degrees(row['psi_rad']):.2f} deg"
    )


In [ ]:
dl1_result = plot_event_cameras(
    event,
    visualizer=visualizer,
    image_level="dl1",
    include_non_triggered=False,
    show_hillas=True,
)
dl1_result["figure"]


## 6. Shower Reconstruction

This runs the same pylast `ShowerProcessor` style as the original simtelarray workflow, but on the LACT ROOT adapter event. The summary prints truth and reconstructed values when available. If an event has too few useful telescopes, `is_valid` can be false; that is a reconstruction-input limitation, not another ROOT read.


In [ ]:
shower_process_config = """{
  "ShowerProcessor": {
    "GeometryReconstructionTypes": [
      "HillasReconstructor"
    ],
    "HillasReconstructor": {
      "ImageQuery": "hillas_intensity > 100 && leakage_intensity_width_2 < 0.3"
    }
  }
}
"""

shower_processor = step(
    "ShowerProcessor init",
    lambda: ShowerProcessor(source.subarray, config_str=shower_process_config),
)
step("shower_processor(event)", lambda: shower_processor(event))

print("DL2 geometry keys:", list(event.dl2.geometry.keys()))
summary = reconstruction_summary(event, "HillasReconstructor")
for key, value in summary.items():
    if isinstance(value, float):
        print(f"{key}: {value:.6g}")
    else:
        print(f"{key}: {value}")


## 7. Cleaned Event With Hillas After Reconstruction

This is intentionally the same event object after both image cleaning and shower reconstruction, so the camera/Hillas view and printed DL2 numbers refer to the same data.


In [ ]:
dl1_hillas_result = plot_event_cameras(
    event,
    visualizer=visualizer,
    image_level="dl1",
    include_non_triggered=False,
    show_hillas=True,
    only_hillas_tels=False,
    show_ideal_position=True,
)
dl1_hillas_result["figure"]


## 8. Optional Save PNGs

Saving is optional. It still uses the already loaded event; no ROOT re-read happens here.


In [ ]:
OUTPUT_DIR = ROOT_FILE.parent / "pylast_visualize"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plots = {
    "core": core_result,
    "sdp": sdp_result,
    "dl0_cameras": dl0_result,
    "dl1_hillas": dl1_hillas_result,
}
for name, result in plots.items():
    figure = result.get("figure")
    if figure is None:
        continue
    path = OUTPUT_DIR / f"event_{event.event_id}_{name}.png"
    figure.savefig(path, dpi=200, bbox_inches="tight")
    print(path)


## Server Notes

Typical setup after pulling the two repositories:

```bash
cd /home/lhaaso/huangyiyun/LACT/Sim_program/pylast
git pull yun lact_sim
export CONDA_PKGS_DIRS=/home/lhaaso/huangyiyun/conda/pkgs
export MPLCONFIGDIR=/home/lhaaso/huangyiyun/tmp/matplotlib
python -m pip install -e . --no-build-isolation
jupyter lab notebooks/lact_sim_root_quicklook.ipynb
```

If ROOT is not on the default linker path, load or export the same ROOT used when pylast was built before launching Jupyter.
